# Check city boundaries

This notebook loops through cities and plots their city boundaries.

## Parameters

In [ ]:
%run -i "functions.py"
cityfilename = "european_capitalsand100000pop.csv"
boundarycheck_quick = True # If True, do a quick check, otherwise save large maps with basemaps

## Load cities

In [ ]:
with open('../cities/'+cityfilename, mode='r') as infile:
    reader = csv.reader(infile, delimiter=";")
    header = next(reader)
    cities = {slugify(rows[0]): {header[0]: rows[0], header[1]: rows[1], header[2]: rows[2]} for rows in reader}
os.makedirs("../plots/", exist_ok=True)

## Check city boundaries

In [ ]:
for cityid, city_info in cities.items():
    # Draw location polygons and their holes
    if city_info["nominatim_query"]:
        location = ox.geocoder.geocode_to_gdf(city_info["nominatim_query"])
        location = fill_holes(extract_relevant_polygon(cityid, shapely.geometry.shape(location['geometry'][0])))
    else:
        # https://github.com/mszell/bikenwgrowth/blob/main/code/01_prepare_networks.ipynb
        shp = gpd.read_file("../cities/"+cityid+".shp")
        location = shp.iloc[0].geometry

    if boundarycheck_quick:
        fig = plt.figure(figsize=(2,2))
    else:
        fig = plt.figure()
    ax = fig.add_axes([0,0,1,1])
    try:
        color = cm.rainbow(np.linspace(0,1,len(location)))
        for poly,c in zip(location, color):
            plt.plot(*poly.exterior.xy, c = c)
            for intr in poly.interiors:
                plt.plot(*intr.xy, c = "red")
    except:
        plt.plot(*location.exterior.xy)
    if not boundarycheck_quick: 
        contextily.add_basemap(ax=ax, url=contextily.providers.CartoDB.Positron, crs='EPSG:4326')
        plt.gca().set_aspect('equal')
    ax.set_title(city_info["name_en"]+", "+city_info["country_en"])
    plt.show()
    if not boundarycheck_quick:
        # ax.autoscale(enable=True, tight=True)
        fig.savefig(f"../plots/boundary_{cityid}.png", dpi=150, bbox_inches='tight')
        plt.close()